In [15]:
import os
import glob
from pathlib import Path
from vip_slap2_analysis.plotting.movies import (
    MovieRenderConfig,
    load_slap2_movie_from_tiffs,
    render_glutamate_df_movie,
    inspect_tiff_layout,
)
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.glutamate.summary import GlutamateSummary

In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SESSION_ID = None  # e.g. "834788_2026-03-04_08-43-07"; leave as None to use SUBJECT_ID + SESSION_INDEX
SUBJECT_ID = 834788
SESSION_INDEX = 6  # used only when SESSION_ID is None

FS_HZ = 200.0
DMD_FOR_CALCIUM = 2
ROI_INDICES_TO_PLOT = [0,1,2]          # whole-session calcium traces
MAX_SESSION_MINUTES = None         # e.g. 10.0 to truncate late-session data
MOTION_CORRECT = True
TRACE_TYPE = "Fsvd"
CALCIUM_TRACE_KEY = "dff"         # one of: dff, ca_mc, ca_unmixed, ca_clean

DMD_FOR_MEANIM = 2
MEANIM_IMAGE_TYPE = "meanIM"
MEANIM_CHANNEL = 0

In [6]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

if SESSION_ID is not None:
    session_row = registry.get_session_row(SESSION_ID)
else:
    subject_sessions = registry.sessions(subject_ids=[SUBJECT_ID]).reset_index(drop=True)
    if len(subject_sessions) == 0:
        raise ValueError(f"No sessions found for subject {SUBJECT_ID}")
    session_row = subject_sessions.iloc[SESSION_INDEX]

asset = registry.resolve_assets(session_row)
asset.ensure_dirs()
exp = GlutamateSummary(asset.summary_mat, keep_open=True)

print(f"session_id: {asset.session_id}")
print(f"subject_id: {asset.subject_id}")
print(f"summary_mat: {asset.summary_mat}")
print(f"bonsai_csv: {asset.bonsai_event_log_csv}")
print(f"qc_dir: {asset.qc_dir}")
print(f"derived_dir: {asset.derived_dir}")

session_id: 834788_2026-03-19_09-05-56
subject_id: 834788
summary_mat: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\834788\834788_2026-03-19_09-05-56\834788_2026-03-19_09-05-56\slap2\dynamic_data\ExperimentSummary\SummaryLoCo-260320-073439.mat
bonsai_csv: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\834788\834788_2026-03-19_09-05-56\834788_2026-03-19_09-05-56\behavior\VCO1_Behavior.harp\bonsai_event_log.csv
qc_dir: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\834788\834788_2026-03-19_09-05-56\analysis\qc
derived_dir: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\834788\834788_2026-03-19_09-05-56\analysis\derived


In [7]:
asset.session_dir

WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/834788/834788_2026-03-19_09-05-56')

In [8]:
cfg = MovieRenderConfig(
    input_frame_rate_hz=80.0,
    output_frame_rate_hz=10.0,
    downsample_factor_time=0,
    baseline_window_s=0.25,
    channel_index=0,
    padding_px=30,
    activity_percentile=99.5,
    structure_percentile=99.5,
    gamma=1.4,
)

In [9]:
trial = 12

dmd1_path = glob.glob(os.path.join(asset.session_dir,'**',
                                   f'E1T{trial}DMD1_REGISTERED_DOWNSAMPLED-80Hz.tif'),
                      recursive=True)[0]

dmd2_path = glob.glob(os.path.join(asset.session_dir,'**',
                                   f'E1T{trial}DMD2_REGISTERED_DOWNSAMPLED-80Hz.tif'),
                      recursive=True)[0]

In [10]:
inspect_tiff_layout(dmd1_path)

path: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\834788\834788_2026-03-19_09-05-56\834788_2026-03-19_09-05-56\slap2\dynamic_data\E1T12DMD1_REGISTERED_DOWNSAMPLED-80Hz.tif
series count: 1
series 0: shape=(1494, 1021, 696), axes=IYX
pages: 1494
memmap shape: (1494, 1021, 696) dtype: float32


In [ ]:
movie = load_slap2_movie_from_tiffs(dmd1_path, dmd2_path, n_channels=2)
print(movie.shape)

In [ ]:
output_path = os.path.join(basepath,'test',"session_summary_movie.mp4")

In [ ]:
render_glutamate_df_movie(
    output_path="debug_crop_5s.mp4",
    config=cfg,
    dmd1_tiff=dmd1_path,
    dmd2_tiff=dmd2_path,
    crop_bbox=(300, 900, 250, 1100),
    start_s=0,
    end_s=5,
    overwrite=True,
)